# Mixed layer heat budget analysis of ACCESS-OM2 runs

This notebook contains code to analyse the mixed layer temperature budget in ACCESS-OM2 for a given time period of interest.

This notebook relies on pre-computed grouped budget quantities as computed by the budget processing scripts in this repository (e.g. `

The theory behind this budget and the diagnostics used to analyse it are summarized in the README.md of this repository. More details can be found in a publication in preparation (TBC).

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=4)
client

In [ ]:
# Chdir for figure saving:
os.chdir('access-om2-analysis/access-om2-sst-budget')

# Load data

### Define paths, region to analyse and time period to analyse

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
output = 364 # 364 = 2017
#output = 365 # 365 = 2018
#output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors

tmp_folder = base + 'post_processed_diags/'

base2 = base + 'output%03d/ocean/' % output

# Climatology:
clim_str = 'output336-365' # 336-365 = 1989-2018
clim_label = '1989-2018'

# Subsample regions:
#reg = [-100, 20, 0, 70] # North Atlantic
#reg = [-100, -40, 0, 30] # North Atlantic
#reg = [-230, -190, -50, -10] # EAC
reg = [135-360,175-360, -60, -20] # SE Aus (Kajtar et al. 2022)
#reg = [None,None,None,None] # Globe
#reg = [-270, -70, -60, 60] # Pacific
#reg = [-270, -210, -20, 20] # Maritime continent
#reg = [-270, -180, -45, 0] # Australia

# Subsample time:
#times = slice('2019-01-01','2019-01-31')#lice(None,None)
#times_snap = slice('2019-01-01','2019-02-01') # Note; this must be 1 more than times.
times = slice('2017-09-01',None)
times_snap = slice('2017-09-01',None) # Note; this must be 1 more than times.
#times = slice(None,None)
#times_snap = slice(None,None) # Note; this must be 1 more than times.

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

### Load grid and set constants

In [ ]:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)
rho0 = 1035.
Cp = 3992.10322329649

### Load daily data

#### Standard variables:

In [ ]:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

#### Pre-computed MLT budget - standard averaging
These files contain pre-computed grouped MLT budget terms. Computed using `daily_online_mlt_budget_year.sub` and `spawn_daily_online_mlt_budget_year.py` and the functions defined below.

The resulting dataset `mlt_budget_stavg_daily` will contain x*y*t arrays with the above budget groups (e.g. advection, surface forcing etc.) as well as:
- fixedh\_tendency: The tendency term of mixed layer temperature with a fixed ML depth (i.e. the sum of all the RHS terms listed above, including the correction terms).
- residual: The residual, fixedh\_tendency minus all the RHS terms. This should be exactly zero. If it isn't, you're missing terms in the source MOM5 budget or something has gone wrong.
- mlt\_tendency: The tendency of the mixed layer temperature computed (in this case) from snapshots of the mixed layer temperature (the diagnostic temp\_in\_mld) at the start and end of the day.
- entrainment: The entrainment term, computed by residual mlt\_tendency - fixedh\_tendency (with the later here equal to the sum of the RHS terms, so the mlt budget closes)

All terms have units of degC/second.

We also optionally load a climatology of the same budget files. Note: to compute these, just use `ncea *.nc output.nc` from the command line, including all the years desired.

In [ ]:
# Pre-computed standard average online daily
mlt_budget_stavg_daily = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_online_output%03d.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times)

In [ ]:
# Pre-computed standard average online climatology (1989-2018, outputs 336-365):
mlt_budget_stavg_clim = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_online_' + clim_str + '_monthly_mean.ncea.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

#### Load climatologies of selected standard variables

These are pre-computed, see below.

In [ ]:
# Standard average daily budget diagnostics:
ds_clim = xr.open_dataset(tmp_folder + 'ocean_month_' + clim_str + '.clim.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim_snapshot = xr.open_dataset(tmp_folder + 'ocean_snapshot_month_' + clim_str + '.clim.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_clim = ds_clim.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim.time.values]})
ds_clim_snapshot = ds_clim_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim_snapshot.time.values]})

# Add wrap-around for clim_snapshot:
tminus1 = str(ds_clim_snapshot.time.isel(time=-1).values)
tminus1 = np.datetime64(str(int(tminus1[:4])-1) + tminus1[4:])
ds_clim_snapshot = xr.concat([ds_clim_snapshot.isel(time=-1).assign_coords({'time':tminus1}),ds_clim_snapshot],dim='time')

### Compute climatologies for selected standard variables (only needed if not already computed):

In [ ]:
outputs = np.arange(336,366)                                           # Define outputs to include in climatology
fields = {'ocean_snapshot_month.nc':'all',                             # Define variables to compute
          'ocean_month.nc':['temp_in_mld','salt_in_mld','mld']
         }

dest_post = '_output%03d-%03d.clim.nc' % (outputs[0],outputs[-1])      # postfix for files

In [ ]:
for file in fields.keys():
    print('Doing ' + file + '...')
    ds = xr.open_dataset(base + 'output%03d' % outputs[0] + '/ocean/' + file,decode_times=False)
    if fields[file] != 'all':
        ds = ds[fields[file]]
    ds.load()

    for output in tqdm(outputs[1:]):
        ds2 = xr.open_dataset(base + 'output%03d' % output + '/ocean/' + file,decode_times=False)
        if fields[file] != 'all':
            ds2 = ds2[fields[file]]
        ds2.load()
        ds2 = ds2.assign_coords({'time':ds.time})
        ds = ds + ds2

    ds = ds/len(outputs)
    ds.to_netcdf(tmp_folder + file.replace('.nc',dest_post))        

### Raw daily mld-binned budget variables and snapshots (only if computing/recomputing new groups or wanting to plot snapshots):

In [ ]:
# Standard average daily budget diagnostics:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Falling average daily budget daignostics (while the name of the averaging is "risavg", in effect this is actually the falling average diagnostics):
ds_day_budget_falavg = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
ds_day_budget_falavg = ds_day_budget_falavg.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_falavg.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget.average_DT.data = ds_day_budget.average_DT*np.timedelta64(1,'D')
ds_day_budget_falavg.average_DT.data = ds_day_budget_falavg.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_day_budget = ds_day_budget.sel(time=times)
ds_day_budget_falavg = ds_day_budget_falavg.sel(time=times)

In [ ]:
# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

### 3D budget data (only used for online-offline error comparison)

In [ ]:
# # monthly data:
ds_mon_budget_3d = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_budget_3d = xr.open_dataset(base2 + 'ocean_budget_daily_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Snapshots for standard average tendency computation:
ds_mon_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_mon_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon_snapshot = xr.concat([ds_mon_snapshot_m1.isel(time=-1),ds_mon_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_mon_budget_3d = ds_mon_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget_3d.time.values]})
ds_day_budget_3d = ds_day_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_3d.time.values]})
ds_mon = ds_mon.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon.time.values]})
ds_mon_snapshot = ds_mon_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_mon_budget_3d.average_DT.data = ds_mon_budget_3d.average_DT*np.timedelta64(1,'D')
ds_day_budget_3d.average_DT.data = ds_day_budget_3d.average_DT*np.timedelta64(1,'D')
ds_mon.average_DT.data = ds_mon.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_mon_budget_3d = ds_mon_budget_3d.sel(time=times)
ds_day_budget_3d = ds_day_budget_3d.sel(time=times)
ds_mon = ds_mon.sel(time=times)
ds_mon_snapshot = ds_mon_snapshot.sel(time=times_snap)

# Analyse a specific event/region

This section contains various plotting features to analyse an event of interest.

First, define the region of interest (for spatial averaging) and its name:

In [ ]:
sreg = [150-360,168-360,-44,-38] # Tasman Sea region from Kajtar et al. 2022
region_name = 'Tasman Sea'
#sreg = [-70,5,0,60] # North Atlantic region
#region_name = 'North Atlantic'

## Plot time series averaged over the event region

Then, compute a daily-resolution climatology covering the period of interest

In [ ]:
add_extra = True # Whether to add a second year in the climatologies in order to cover the second half of December

year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])
year_clim = int(str(ds_clim.time[0].astype('datetime64[Y]').values)[:4])

# climatology, standard variables:
ds_climA = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in ds_clim.time]})
if add_extra:
    ds_clim2 = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in ds_clim.time]})
    ds_climA = xr.concat([ds_climA,ds_clim2],dim='time')
ds_climA = ds_climA.resample(time='1D').interpolate("linear").sel(time=times)

# Climatology, budget variables:
mlt_budget_stavg_climA = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in mlt_budget_stavg_clim.time]})
if add_extra:
    mlt_budget_stavg_clim2 = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in mlt_budget_stavg_clim.time]})
    mlt_budget_stavg_climA = xr.concat([mlt_budget_stavg_climA,mlt_budget_stavg_clim2],dim='time')
mlt_budget_stavg_climA = mlt_budget_stavg_climA#.resample(time='1D').interpolate("linear").sel(time=times)

Finally, plot the time series

In [ ]:
fig, axes = plt.subplots(nrows=4,ncols=1,figsize=(12,16),height_ratios=[1.,0.5,1.,1.])

# Panel 1: Mixed layer temperature, including climatology and snapshots:
mlt = (ds_day.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt_snap = (ds_day_snapshot.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt.plot(ax=axes[0],label='Daily-average mixed layer temperature',linewidth=5.)
mlt_snap.plot(ax=axes[0],label='Snapshot mixed layer temperature',linewidth=2.,linestyle='dashed')
(ds_climA.temp_in_mld/rho0).plot(ax=axes[0],label=clim_label + ' climatology',linewidth=2.)

# # Plot some averages:
# Octavg = mlt.sel(time=slice('2017-10-01','2017-10-31')).mean('time')
# Decavg = mlt.sel(time=slice('2017-12-01','2017-12-31')).mean('time')
# axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-11-01')],[Octavg.values,Octavg.values],'-',color='C0',linewidth=2.)
# axes[0].plot([np.datetime64('2017-12-01'),np.datetime64('2018-01-01')],[Decavg.values,Decavg.values],'-',color='C0',linewidth=2.)
# axes[0].plot([np.datetime64('2017-10-16T12:00:00'),np.datetime64('2017-12-16T12:00:00')],[Octavg.values,Decavg.values],'-',color='C0',linewidth=2.,linestyle='dashed',label='Epoch difference (Dec minus Oct)')
# axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-12-31')],[mlt_snap.sel(time='2017-10-01',method='nearest'),mlt_snap.sel(time='2017-12-31',method='nearest')],'-',color='C1',linewidth=2.,linestyle='dotted',label='Snapshot difference (24Z 31st Dec - 24Z 1st Oct)')

axes[0].legend()
axes[0].set_title(region_name + ' mixed layer temperature budget')
axes[0].set_ylabel('Temperature ($\circ$C)')
axes[0].grid()

# Panel 2: Mixed layer depth and climatology:
ds_day.mld.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).plot(ax=axes[1],linewidth=2.,label='Mixed layer depth')
(ds_climA.mld).plot(ax=axes[1],label=clim_label + ' climatology',linewidth=2.)
axes[1].set_ylabel('Mixed layer depth (m)')
axes[1].grid()
axes[1].legend()
axes[1].set_ylim([0.,150.])

# Panel 3: Budget terms (raw)
vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']

budget_stavg = mlt_budget_stavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
#budget_hatavg = mlt_budget_hatavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
budget_stavg_clim = mlt_budget_stavg_climA
unit_conv = 86400

for j, var in enumerate(vars):
    if j == 0:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j] + ' (st. avg.)')
#        (budget_hatavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,linestyle='dashed',label=labels[j] + ' (hat avg.)')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1.,label=labels[j] + ' (' + clim_label + ' climat.)')
    else:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j])
#        (budget_hatavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,linestyle='dashed')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1)
axes[2].legend()
axes[2].set_ylabel('Temperature tendency ($\circ$C/day)')
axes[2].grid()

# Panel 4: Budget terms (anomalies):

budget_stavg_clim_daily = mlt_budget_stavg_climA.interp(time=budget_stavg.time.astype('datetime64[s]'))
for j, var in enumerate(vars):
    ((budget_stavg[var]-budget_stavg_clim_daily[var])*unit_conv).plot(ax=axes[3],color='C' + str(j),linewidth=2,label=labels[j])
axes[3].legend()
axes[3].set_ylabel('Anomalous temperature \n tendency ($\circ$C/day)')
axes[3].grid()

for ax in axes:
    ax.set_xlabel('')
    ax.set_xlim([mlt.time[0],mlt.time[-1]])
    
#plt.savefig('MLT_budget_' + region_name.replace(' ','') + '_time_series_with_anomalies.png',dpi=250,bbox_inches='tight')

## Plot spatial plot of mixed layer temperature anomalies during event

Currently this is just for the Tasman Sea 2017 event

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(12,3.6),layout='constrained')

mlt = (ds_day.temp_in_mld.resample(time='1ME').mean()/rho0)
mlt_clim = (ds_clim.temp_in_mld/rho0)

(mlt.sel(time='2017-10').drop_vars(['time'])-mlt_clim.sel(time='1989-10').drop_vars(['time'])).plot(ax=axes[0],add_colorbar=False)
(mlt.sel(time='2017-11').drop_vars(['time'])-mlt_clim.sel(time='1989-11').drop_vars(['time'])).plot(ax=axes[1],add_colorbar=False)
(mlt.sel(time='2017-12').drop_vars(['time'])-mlt_clim.sel(time='1989-12').drop_vars(['time'])).plot(ax=axes[2],cbar_kwargs={'label': 'Mixed layer temperature \n anomaly ($^\circ$C)'})

axes[0].set_title('October 2017')
axes[1].set_title('November 2017')
axes[2].set_title('December 2017')
for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
axes[1].set_yticklabels([])
axes[2].set_yticklabels([])

#plt.savefig('MLT_budget_TasmanSea_OctDec2017_MLTA.png',dpi=300,bbox_inches='tight')

## Spatial plots of time-averaged budgets

This section plots spatial plots of the different contributions to budgets integrated over different time periods

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=6, figsize=(20,9))
axs = axes.reshape(-1)

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
clim = 10

# Standard budget difference terms averaged over 3 month period (equivalent to snapshot difference):
times = slice('2017-10-01','2017-12-31')
stavg_budget = mlt_budget_stavg_daily_monthly.sel(time=times).sum('time')

for j, var in enumerate(vars):
    stavg_budget[var].where(stavg_budget[var]!=0.).plot(ax=axes[0][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[0][j].set_title('St. Avg ' + labels[j] + ' ($^\circ$C)')
axes[0][0].set_title(axes[0][0].get_title() + '\n (24Z 31st Dec - 24Z 1st Oct snapshot difference)')

# Hat average budget difference terms between 1st and last month (equivalent to time-average difference):
mlt_monthly = (ds_day.temp_in_mld/rho0).resample(time='1M').mean()
hatavg_budget = monthly_hat_difference(mlt_budget_risavg_monthly.sel(time=times),mlt_budget_stavg_monthly.sel(time=times),mlt_monthly.sel(time=times),0,len(mlt_budget_risavg_monthly.time.sel(time=times))-1)

for j, var in enumerate(vars):
    hatavg_budget[var].plot(ax=axes[1][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[1][j].set_title('Hat. Avg ' + labels[j] + ' ($^\circ$C)')
axes[1][0].set_title(axes[1][0].get_title() + '\n (Dec - Oct average difference)')

# Hat average budget difference terms between 1st and last day (equivalent to time-average difference) as a check:
mlt_daily = (ds_day.temp_in_mld/rho0)
hatavg_budget_daily = monthly_hat_difference(mlt_budget_risavg.sel(time=times),mlt_budget_stavg.sel(time=times),mlt_daily.sel(time=times),0,len(mlt_budget_risavg.time.sel(time=times))-1)

for j, var in enumerate(vars):
    hatavg_budget_daily[var].plot(ax=axes[2][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[2][j].set_title('Hat. Avg ' + labels[j] + ' ($^\circ$C)')
axes[2][0].set_title(axes[2][0].get_title() + '\n (31st Dec - 1st Oct average difference)')

for ax in axs:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
    
plt.tight_layout()
#plt.savefig('MLT_budget_TasmanSea_OctDec2017_spatial.png',dpi=250,bbox_inches='tight')

## Spatial plots of time-averaged online budget anomalies

Currently this is just for the North Atlantic 2023 event used in Matt's paper

In [ ]:
year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])
budget = mlt_budget_stavg_daily.resample(time='1ME').mean() # Take monthly mean
budget = budget.assign_coords({'time':budget.time.astype('datetime64[M]')}) # Replace time stamp with year-month only (to match climatology)
budget = budget - mlt_budget_stavg_clim.assign_coords({'time':[np.datetime64(str(year) + '-01')+np.timedelta64(x,'M') for x in range(12)]}) # Subtract climatology

In [ ]:
sreg = [-100, 20, 0, 60] # North Atlantic region
region_name = 'North Atlantic'
budget_av = (budget*ds_grid.area_t).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])/ds_grid.area_t.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12,6))
axs = axes.reshape(-1)

#times = ['2023-06','2023-07']
#time_labels = ['June 2023','July 2023']
times = ['2023-05','2023-06']
time_labels = ['May 2023','June 2023']
vars = [['mlt_tendency'],
        ['surface_flux','sw_pen'],
        ['advection','vert_mixing','entrainment']]
names = ['MLT tendency','Net surface flux term','Advection, mixing and entrainment']
clims = [-2.4,2.4]
unit_conv = 86400*30.5

for i,time in enumerate(times):
    budget_ti = budget.sel(time=np.datetime64(time,'M'))*unit_conv

    for j, varg in enumerate(vars):
        ds = budget_ti[varg[0]]
        if len(varg)>1:
            for k in range(len(varg)-1):
                ds += budget_ti[varg[k+1]]
        ds.plot.contourf(ax=axes[i][j],levels=np.arange(-2.4,2.8,0.4),cmap='RdBu_r')
        axes[i][j].set_title(names[j] + ' (' + time_labels[i] + ')')
        axes[i][j].set_xlabel('')
        axes[i][j].set_ylabel('')
        axes[i][j].set_facecolor([0.5,0.5,0.5])
        axes[i][j].set_xlim(reg[:2])
        axes[i][j].set_ylim([reg[2],reg[3]])

axes[0][0].plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
    
plt.tight_layout()
plt.savefig('MLT_budget_NorthAtlantic_spatial_May_Jun.png',dpi=200,bbox_inches='tight')

## Bar plots of region averages:

Again, just for the North Atlantic case currently

In [ ]:
fig = plt.figure(figsize=(12,15))
ax = plt.gca()

times = ['2023-05','2023-06','2023-07','2023-08']
time_labels = ['May','June','July','August']
vars = [['mlt_tendency'],
        ['surface_flux','sw_pen'],
        ['shortwave','sw_pen'],
        ['latent'],
        ['longwave'],
        ['sensible'],
        ['vert_mixing','entrainment'],
       ['advection']]
names = ['MLT tendency','Net surface flux','Shortwave','Latent','Longwave','Sensible','Vertical mixing and entrainment','Advection']

# Create variable groups:
budget_av_gr = budget_av['mlt_tendency'].rename(names[0]).to_dataset()
for i, varg in enumerate(vars[1:]):
    budget_av_gr[names[i+1]] = budget_av[varg[0]]
    if len(varg)>1:
        for k in range(len(varg)-1):
            budget_av_gr[names[i+1]] += budget_av[varg[k+1]]
    
df = budget_av_gr.rename({'time':'class'}).assign_coords({'class':time_labels}).to_dataframe().reset_index()
   
# Parameters
num_vars = len(times)
num_classes = len(names)
bar_width = 0.1
x = np.arange(num_vars)  # One x position per variable

unit_conv = 86400*30.5
# Create figure

# Plot each class as a separate bar group
for i, cls in enumerate(names):
    # Get values for this class across all variables
    values = list(df[cls]*unit_conv)
    
    # Offset x positions for each class
    ax.bar(x + i * bar_width, values, width=bar_width, label=cls)

# Formatting
ax.set_xticks(x + bar_width)
ax.set_xticklabels(time_labels)
ax.set_yticks(np.arange(-0.8,3.2,0.4))
ax.set_ylim([-0.8,1.8])
ax.set_ylabel("$^\circ$C/month")
ax.set_title('North Atlantic MLT budget anomalies 2023')
ax.legend(title="Budget terms",fontsize=8,loc='upper right',bbox_to_anchor=[1.1,0.9,0.1,0.1])
ax.grid()
plt.tight_layout()
plt.savefig('MLT_budget_NorthAtlantic_time_series_big.png',dpi=200,bbox_inches='tight')

In [ ]:
budget_av.surface_flux.values

In [ ]:
budget_av.sw_pen.values

In [ ]:
budget_av.mlt_tendency.values

# Define functions to construct grouped Mixed-layer temperature budget terms

Note: This code should not need to be run if grouped MLT budget terms have already been computed.

First we define the term groups

In [ ]:
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}

Then define functions to compute the correction terms, do the grouping and compute the tendency and entrainment terms

In [ ]:
def compute_corrections(ds_day_budget):
    """
    Compute corrections to advection, surface mass flux and total tendency terms in ds_day_budget.

    For the P-E+R correction:
    -------------------------
    
    sfc_hflux_pme_in_mld              =     Ca*Qm/H*Cp                      (Wm-3)
    pme_river_times_temp_in_mld       =     CH*Qm/H                         (kg m-3 deg C s-1)
    sfc_hflux_pme_in_mld_cor          =     (Ca-CH)*Qm/H*Cp                 (Wm-3)

    For the advection correction:
    -----------------------------
    adv_cor1                          =     CH*divU/H                       (deg C s-1), where divU = Qm/rho0 + Ssmoother - deta/dt from the free-surface equation
    adv_cor2                          =     Cent*(1- H/(D+eta))*deta/dt/H   (deg C s-1)

    Produces the corrected terms sfc_hflux_pme_in_mld_cor, temp_eta_smooth_in_mld_cor, temp_advection_in_mld_cor and temp_tendency_in_mld_cor
    """

    # P-E+R correction: 
    pme_cor = -ds_day_budget['pme_river_times_temp_in_mld']/rho0
    ds_day_budget['sfc_hflux_pme_in_mld_cor'] = ds_day_budget['sfc_hflux_pme_in_mld'] + pme_cor*rho0*Cp

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_temp_in_mld']/rho0
    ds_day_budget['temp_eta_smooth_in_mld_cor'] = ds_day_budget['temp_eta_smooth_in_mld'] + eta_smoother_cor*rho0*Cp

    # Advection correction:
    adv_cor1 = (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0
    adv_cor2 = ds_day_budget['s_surf_ent_temp']/rho0
    ds_day_budget['temp_advection_in_mld_cor'] = ds_day_budget['temp_advection_in_mld'] + adv_cor1*rho0*Cp  + adv_cor2*rho0*Cp

    # Total correction for entrainment-by-residual:
    ds_day_budget['temp_tendency_in_mld_cor'] = ds_day_budget['temp_tendency_in_mld']  + pme_cor*rho0*Cp + adv_cor1*rho0*Cp + adv_cor2*rho0*Cp + eta_smoother_cor*rho0*Cp

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    if do_extras=True, also add extra budget terms (e.g. the components of the surface flux) from bud_var_extras.
    """
    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget[bud_tendency]/rho0/Cp).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0/Cp
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0/Cp
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

# Define functions to construct grouped Mixed-layer salinity budget terms

As above, but for salinity

In [ ]:
bud_tendency = 'salt_tendency_in_mld_cor'
bud_var_grps = {'advection':['salt_advection_in_mld_cor',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',       # This is the restoring term - if wanting to separate it out.
                                'pme_in_mld_cor',                 # THis is the surface freshwater flux term - the main one. It enters as a "correction" term in the language of the MLT budget described in the theory above and in the paper.
                                'salt_eta_smooth_in_mld_cor']}
bud_var_extras = {}

In [ ]:
def compute_corrections(ds_day_budget):
    """
    Compute corrections to advection and surface mass flux terms in ds_day_budget for salinity
    """

    # P-E+R correction:
    # pme_river_times_salt_in_mld = psu kg m -3 s-1
    pme_cor = -ds_day_budget['pme_river_times_salt_in_mld']/1000.
    ds_day_budget['pme_in_mld_cor'] = pme_cor

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_salt_in_mld']/1000.
    ds_day_budget['salt_eta_smooth_in_mld_cor'] = ds_day_budget['salt_eta_smooth_in_mld'] + eta_smoother_cor

    # Advection correction:
    adv_cor1 = -(ds_day_budget['eta_t_tendency_times_salt_in_mld']-ds_day_budget['pme_river_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.
    adv_cor2 = ds_day_budget['s_surf_ent_salt']/1000.
    ds_day_budget['salt_advection_in_mld_cor'] = ds_day_budget['salt_advection_in_mld'] + adv_cor1 + adv_cor2

    # Total correction for residual:
    ds_day_budget['salt_tendency_in_mld_cor'] = ds_day_budget['salt_tendency_in_mld']  + pme_cor + adv_cor1 + adv_cor2 + eta_smoother_cor

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    if do_extras=True, also add extra budget terms (e.g. the components of the surface flux) from bud_var_extras.
    """
    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget[bud_tendency]/rho0*1000.).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0*1000.
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0*1000.
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0*1000.
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0*1000.

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

# Compute budget terms

Note: These cells are just for testing, if you're working with pre-computed budget terms, none of this code should be needed.

## daily budget, standard averaging

In [ ]:
# Compute in one go:
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily = compute_tendency_entrainment(mlt_budget_stavg_daily,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily.load();

In [ ]:
# Compute in blocks (e.g. if doing a whole year):

ds_day_budget = compute_corrections(ds_day_budget)

# NOTE: THIS still seems way slower than it should be... Something simple might make it faster...
bs = 30; tl = len(ds_day_budget.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

mlt_budget_stavg_daily_uncat = []
for i in tqdm(range(len(blocks))):
    bud = mlt_budget_fixedh(ds_day_budget.isel(time=blocks[i]))
    bud = compute_tendency_entrainment(bud,ds_day_snapshot.temp_in_mld.isel(time=blocks_snap[i])/rho0)
    mlt_budget_stavg_daily_uncat.append(bud.load())
mlt_budget_stavg_daily = xr.concat(mlt_budget_stavg_daily_uncat,dim='time')

In [ ]:
# Save to file if desired:
mlt_budget_stavg_daily.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_online_output%03d.nc' % output)

## Compute monthly budget climatology

Note: There is currently some bug with the monthly budget's accumulations - temp_tendency_in_mld shows a signature of the daily cycle (zonal cooling in the east pacific and warming zonally elsewhere). I don't know what it is. The daily files seem fine.

### From raw climatology files (e.g. climatology computed before term grouping/entrainment/tendency etc.)

In [ ]:
# Compute fixedh budget:
mlt_budget_stavg_clim = mlt_budget_fixedh(ds_clim_budget)

# Compute entrainment and tendnecy:
mlt_budget_stavg_clim = compute_tendency_entrainment(mlt_budget_stavg_clim,ds_clim_snapshot.temp_in_mld/rho0)

# Force computation:
mlt_budget_stavg_clim.load();

### From pre-computed climatology files (e.g. including pre-computed grouping/entrainment/tendency etc.)

In [ ]:
base = '/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
outputs = np.arange(336,366) # 1989-2018
dest_folder = base + 'clim_1989-2018/'

for output in tqdm(outputs):

    ######################## Load data for this year:

    base2 = base + 'output%03d' % output + '/ocean/'
    
    # Standard average daily budget diagnostics:
    ds_mon_budget = xr.open_dataset(base2 + 'ocean_budget_month.nc',decode_times=False,chunks=chunks2D)
    
    # Snapshots for standard average tendency computation:
    ds_mon_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D)
    # Add previous output for last element:
    ds_mon_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D)
    ds_mon_snapshot = xr.concat([ds_mon_snapshot_m1.isel(time=-1),ds_mon_snapshot],dim='time')
    
    # Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
    ds_mon_budget = ds_mon_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget.time.values]})
    ds_mon_snapshot = ds_mon_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_snapshot.time.values]})
    
    # Fix average_DT by decoding by hand:
    ds_mon_budget.average_DT.data = ds_mon_budget.average_DT*np.timedelta64(1,'D')
    
    # Subselect time period:
    ds_mon_budget = ds_mon_budget.sel(time=times)
    ds_mon_snapshot = ds_mon_snapshot.sel(time=times_snap)

    ############### Compute budget:
    mlt_budget_stavg_monthly = mlt_budget_fixedh(ds_mon_budget)
    mlt_budget_stavg_monthly = compute_tendency_entrainment(mlt_budget_stavg_monthly,ds_mon_snapshot.temp_in_mld/rho0)
    mlt_budget_stavg_monthly.load();

    ############### Save to file:
    mlt_budget_stavg_monthly.to_netcdf(tmp_folder + 'mlt_budget_stavg_monthly_online_output%03d.nc' % output)
    

In [ ]:
mlt_budget_stavg_monthly.isel(time=0).entrainment.plot()

## Compute monthly difference budget, standard averaging
`mlt_budget_stavg_monthly` is simply the time integral of of the daily `mlt_budget_stavg_daily` budget. The resulting array contains the temperature difference induced by each term (or the temperature difference itself, for mlt\_tendency) across the month. Units are degC.

In [ ]:
mlt_budget_stavg_monthly = (mlt_budget_stavg_daily*(ds_day.average_DT/np.timedelta64(1,'s'))).resample(time='1ME').sum()
mlt_budget_stavg_monthly['time'] = mlt_budget_stavg_daily.time.resample(time='1ME').mean() # Centre time in the middle of the month.

In [ ]:
ds = mlt_budget_stavg_monthly.entrainment.load()

In [ ]:
# Another res check:
ds = mlt_budget_stavg_daily['mlt_tendency'] - mlt_budget_stavg_daily['advection'] - mlt_budget_stavg_daily['surface_flux']  - mlt_budget_stavg_daily['vert_mixing']   - mlt_budget_stavg_daily['sw_pen']    - mlt_budget_stavg_daily['entrainment']

In [ ]:
(ds.isel(time=-10)*86400).plot()

In [ ]:
fig, axes  = plt.subplots(nrows=3,ncols=4,figsize=(20,10))
for i, var in enumerate(mlt_budget_stavg_daily.data_vars):
    (mlt_budget_stavg_daily[var]*86400).isel(time=-10).plot(ax=axes.reshape(-1)[i],vmin=-.5,vmax=.5,cmap='RdBu_r')
    axes.reshape(-1)[i].set_title(var)
    #ds.isel(time=mn).plot(ax=axes.reshape(-1)[mn],vmin=-2.,vmax=2.,cmap='RdBu_r')
plt.tight_layout()

## Compute daily budget, hat averaging

Compute the hat-averaged daily budget. This budget (containing the same fields as `mlt_budget_stavg_daily`) corresponds to the tendency of the *daily-averaged* mixed layer temperature. E.g., mlt\_tendency in `mlt_budget_hatavg_daily` is the difference in daily-averaged temperature between the day after and the day before the time stamp (with the time stamp being at 0Z inbetween the two days), divided by the number of seconds in the day (units degC/second). The other terms are the budget contributions to this tendency. 

In [ ]:
def hat_average(st_avg,fal_avg_raw,average_DT):
    """
    Compute hat average from standard and falling average for a particular field
    """
    fal_avg = fal_avg_raw/(average_DT/np.timedelta64(1,'s')) # This line fixes a bug in the normalization of the daily falling average diagnostics
                                                             # take mean by dividing by averaging period. We do this here to avoid loading all the variables just to do this fix.
    ris_avg = st_avg - fal_avg                               # Rising = standard - falling
    hat_avg = ris_avg.isel(time=slice(0,-1)).values +  fal_avg.isel(time=slice(1,None)) # Hat = rising over first day - falling over second day              
                                                                                        # Note: Dealing with time is done outside this function
    return(hat_avg)

In [ ]:
# Template variable (note that hat average tendencies lie on snapshot (1:end-1) time:
mlt_budget_hatavg_daily = np.nan*xr.zeros_like(ds_day_snapshot.temp_in_mld.isel(time=slice(1,-1)).transpose(*ds_day_budget['temp_tendency_in_mld'].dims)) 

# Compute hat average fixedh tendency:
mlt_budget_hatavg_daily.data = hat_average(ds_day_budget['temp_tendency_in_mld'],ds_day_budget_falavg['temp_tendency_in_mld'],ds_day_budget_falavg.average_DT)/rho0/Cp

# Make a dataset:
mlt_budget_hatavg_daily = mlt_budget_hatavg_daily.rename('fixedh_tendency').to_dataset()

# Do other variables:
for var in bud_var_grps.keys():
    mlt_budget_hatavg_daily[var] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
    mlt_budget_hatavg_daily[var].data = hat_average(ds_day_budget[bud_var_grps[var][0]],ds_day_budget_falavg[bud_var_grps[var][0]],ds_day_budget_falavg.average_DT)/rho0/Cp
    if (len(bud_var_grps[var])>1):
        for raw_var in bud_var_grps[var][1:]:
            mlt_budget_hatavg_daily[var].data += hat_average(ds_day_budget[raw_var],ds_day_budget_falavg[raw_var],ds_day_budget_falavg.average_DT)/rho0/Cp

# Compute residual for check:
mlt_budget_hatavg_daily['residual'] = mlt_budget_hatavg_daily['fixedh_tendency'].copy(deep=True)
for var in list(mlt_budget_hatavg_daily.data_vars):
    mlt_budget_hatavg_daily['residual'] -= mlt_budget_hatavg_daily[var]

# Compute mlt tendency:
mlt = (ds_day.temp_in_mld/rho0).transpose(*mlt_budget_hatavg_daily['fixedh_tendency'].dims)
mlt_budget_hatavg_daily['mlt_tendency'] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
mlt_budget_hatavg_daily['mlt_tendency'].data = mlt.isel(time=slice(1,None)).values - mlt.isel(time=slice(0,-1)).values
mlt_budget_hatavg_daily['mlt_tendency'] = mlt_budget_hatavg_daily['mlt_tendency']/86400. # Note: this will only work for daily averaging, 
                                                                                           # as it assumes a 86400 time difference between 
                                                                                           # the centre of the two time-averaged periods (day before to day after)

# Entrainment term by residual:
mlt_budget_hatavg_daily['entrainment'] = -(mlt_budget_hatavg_daily['fixedh_tendency'] - mlt_budget_hatavg_daily['mlt_tendency'])

## Compute monthly difference budget, hat averaging

This section computes the monthly difference budgets (e.g. as above, contributions to the temperature differences across individual months) for standard averaging and hat averaging from the daily budgets. It then defines a function `monthly\_hat\_average` that takes these budgets as inputs, along with the monthly-averaged mixed layer temperature and two month indexes of interest, and outputs the contributions to the budget that govern the difference between the monthly-averaged mixed layer temperature of those two months.

Note: Some code and calculations here are repeated from the daily averaging performed above for conveninence.

In [ ]:
# Daily standard average budget (duplicated from above):
mlt_budget_stavg = mlt_budget_fixedh(ds_day_budget)

# Multiply by Delta t for differences:
mlt_budget_stavg = mlt_budget_stavg*(ds_day_budget.average_DT/np.timedelta64(1,'s'))

In [ ]:
# Daily falling average budget:
mlt_budget_falavg = mlt_budget_fixedh(ds_day_budget_falavg)

In [ ]:
# Daily rising average:
mlt_budget_risavg = mlt_budget_stavg - mlt_budget_falavg

In [ ]:
# Define n-1 DataArrays for the months:
n_minus_1 = xr.DataArray(data=[x-1 for x in mlt_budget_risavg.time.dt.day.values],dims=['time'],coords={'time':mlt_budget_stavg.time})

# Monthly rising average:
mlt_budget_risavg_monthly = mlt_budget_risavg.resample(time='1ME').mean() + (mlt_budget_stavg*n_minus_1).resample(time='1ME').mean()

# Monthly standard average:
mlt_budget_stavg_monthly = mlt_budget_stavg.resample(time='1ME').sum()

In [ ]:
# Define function to compute full budget given two months of interest:
def monthly_hat_difference(mlt_budget_risavg_monthly,mlt_budget_stavg_monthly,mlt_monthly,month1_index,month2_index):
    """
    Compute hat difference mlt budget from month 1 to month 2. 

    Inputs:
    - mlt_budget_risavg_monthly: The monthly difference budget, rising average
    - mlt_budget_stavg_monthly: The monthly difference budget, standard average
    - mlt_monthly: The monthly-average mixed layer temperature
    - month1_index: The index of the first month
    - month2_index: The index of the second month
    """

    # List of variables:
    vars = list(mlt_budget_risavg_monthly.data_vars)

    # Interim months:
    monthM_index = np.arange(month1_index+1,month2_index,1)
    
    # Compute mlt difference as "mlt_tendency":
    mlt_budget_hat_diff = (mlt_monthly.isel(time=month2_index) - mlt_monthly.isel(time=month1_index)).rename('mlt_tendency').to_dataset()

    # Compute falling difference as difference between other budgets:
    mlt_budget_falavg_monthly = mlt_budget_stavg_monthly - mlt_budget_risavg_monthly

    # Compute budget terms:
    for var in vars:
        mlt_budget_hat_diff[var] = mlt_budget_risavg_monthly[var].isel(time=month1_index) + mlt_budget_stavg[var].isel(time=monthM_index).sum('time') + mlt_budget_falavg_monthly[var].isel(time=month2_index)

    # Compute entrainment by residual:
    mlt_budget_hat_diff['entrainment'] = -(mlt_budget_hat_diff['fixedh_tendency'] - mlt_budget_hat_diff['mlt_tendency'])

    return(mlt_budget_hat_diff)

# Compare offline and daily budgets

What diagnostics are needed to do the offline binning:
- Closed 3D heat budget
- `dzt` and `mld` time-averages to bin the 3D diagnostics into the mixed layer.
- Snapshots of `temp_in_mld` for tendency calculation.

## Define function to compute offline mixed layer temperature budget

In [ ]:
def compute_mixed_layer_temperature_budget_offline(ds_budget,ds_budget_2d,mld,dzt):
    """
    Compute mixed layer temperature budget

    Inputs:
    ds_budget -> dataset containing time-averaged 3D budget quantities
    ds_budget_2d -> dataset containing time-averaged 2D (surface layer only) budget quantities (these are just added in without summing)
    mld -> dataarray containing time-averaged mixed layer depth
    dzt -> datarray containing time-averaged grid cell thicknesses
    temp_mld_snap -> datarray containing snapshots of temp*rho0*dzt summed over mld
    mld_snap -> dataarray containing snapshots of mld

    Outputs:
    ds_budget_in_mld -> dataset containing all the "internal" (i.e. not including mlt tendency and entrainment) terms in the diagnosed mixed layer temperature budget (2D), in units of degC/second
    """

    # Sum over mixed layer:
    ds_budget_in_mld = ds_budget.isel(st_ocean=0,drop=True).copy(deep=True)
    for var in list(ds_budget_2d.data_vars):
        ds_budget_in_mld[var] = xr.zeros_like(ds_budget['fixedh_tendency']).isel(st_ocean=0,drop=True).copy(deep=True)
    for ti in range(len(ds_budget.time)): # Loop over time
        dzt_ti = dzt.isel(time=ti).load()       
        dzt_ti_bot = dzt_ti.cumsum('st_ocean')  # Depth (from free surface) of bottom of each cell
        ds_budget_ti = ds_budget.isel(time=ti).load()
        mld_ti = mld.isel(time=ti).load()
        for var in list(ds_budget.data_vars):
            ds_budget_in_mld[var][ti,:,:] = ds_budget_ti[var].where(dzt_ti_bot<mld_ti).sum('st_ocean')   # Include all of cells that lie completely in the mixed layer
            for k in range(len(ds_budget_ti.st_ocean)-1):
                ds_budget_in_mld[var][ti,:,:] += xr.where(np.logical_and(dzt_ti_bot[k,:,:]>mld_ti,dzt_ti_bot[k+1,:,:]<mld_ti),((mld_ti-dzt_ti_bot[k,:,:])/dzt_ti[k,:,:])*ds_budget_ti[var][k,:,:],0.) # Include only a fraction of cells that lie partially within the mixed layer
            ds_budget_in_mld[var][ti,:,:] = ds_budget_in_mld[var][ti,:,:]/rho0/Cp/mld_ti # Convert units from Wm-2 to degC/sec

        # Add 2D surface layer variables:
        surf_frac = xr.where(dzt_ti_bot[0,:,:]>mld_ti,mld_ti/dzt_ti[0,:,:],1.)            # In regions where the mixed layer depth is shallower than the thickness of the surface grid cell, take only that fraction from the 2D variables
        for var in list(ds_budget_2d.data_vars):
            ds_budget_in_mld[var][ti,:,:] = (surf_frac*ds_budget_2d[var][ti,:,:]/rho0/Cp/mld_ti).load()

    # Compute residual for check:
    ds_budget_in_mld['residual'] = ds_budget_in_mld['fixedh_tendency'] - ds_budget_in_mld[list(ds_budget_in_mld.data_vars)[1:]].to_array().sum('variable')
        
    return(ds_budget_in_mld)

## Compute daily and monthly offline binned mixed layer temperature budgets (standard averaging):

In [ ]:
# Offline budget terms grouping:
bud_var_grps = {'advection':['temp_advection','temp_submeso','temp_vdiffuse_k33','neutral_diffusion_temp','neutral_gm_temp'],
                'vert_mixing':['temp_vdiffuse_diff_cbt','temp_nonlocal_KPP'],
                'surface_flux':['temp_vdiffuse_sbc','frazil_3d','temp_rivermix'],
                'sw_pen':['sw_heat']}
bud_2d_vars = ['temp_eta_smooth','sfc_hflux_pme']

In [ ]:
# Compute monthly budget, by month:

# Add sfc_hflux_pme from standard monthly diagnostics file:
ds_mon_budget_3d['sfc_hflux_pme'] = ds_mon['sfc_hflux_pme']

mlt_budget_stavg_monthly_offline_uncat = []

# Loop over month:
for ti in tqdm(range(len(ds_mon_budget_3d.time))):
    
    # Group terms:
    ds_mon_budget_3d_reduced = ds_mon_budget_3d.isel(time=slice(ti,ti+1))['temp_tendency'].rename('fixedh_tendency').to_dataset().copy(deep=True)
    for var in bud_var_grps.keys():
        ds_mon_budget_3d_reduced[var] = ds_mon_budget_3d.isel(time=slice(ti,ti+1))[bud_var_grps[var][0]].load()
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_mon_budget_3d_reduced[var] += ds_mon_budget_3d.isel(time=slice(ti,ti+1))[raw_var].load()

    ds_mon_budget_3d_reduced_2d = ds_mon_budget_3d.isel(time=slice(ti,ti+1))[bud_2d_vars].load()

    # Do monthly computation:
    bud = compute_mixed_layer_temperature_budget_offline(ds_mon_budget_3d_reduced,ds_mon_budget_3d_reduced_2d,ds_mon.mld.isel(time=slice(ti,ti+1)),ds_mon.dzt.isel(time=slice(ti,ti+1)))
    # Add 2D vars to sbc term:
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])

    bud = compute_tendency_entrainment(bud,ds_mon_snapshot.isel(time=slice(ti,ti+2)).temp_in_mld/rho0)

    mlt_budget_stavg_monthly_offline_uncat.append(bud)

mlt_budget_stavg_monthly_offline = xr.concat(mlt_budget_stavg_monthly_offline_uncat,dim='time')

In [ ]:
# Save to file:
mlt_budget_stavg_monthly_offline.to_netcdf(tmp_folder + 'mlt_budget_stavg_monthly_offline_2023.nc')

In [ ]:
%%time
# Compute daily budget, in blocks:
mlt_budget_stavg_daily_offline_uncat = []

bs = 5; tl = len(ds_day_budget_3d.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

for i in tqdm(np.arange(55,len(blocks)+1)):#range(len(blocks))):
    ds_day_budget_3d_reduced = ds_day_budget_3d.isel(time=blocks[i])['temp_tendency'].rename('fixedh_tendency').to_dataset()
    
    for var in bud_var_grps.keys():
        ds_day_budget_3d_reduced[var] = ds_day_budget_3d.isel(time=blocks[i])[bud_var_grps[var][0]]
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_day_budget_3d_reduced[var] += ds_day_budget_3d.isel(time=blocks[i])[raw_var]
                
    ds_day_budget_3d_reduced_2d = ds_day_budget_3d.isel(time=blocks[i])[bud_2d_vars]

    ds_day_budget_3d_reduced.load()
    ds_day_budget_3d_reduced_2d.load()
    mld = ds_day.mld.isel(time=blocks[i]).load()
    dzt = ds_day_budget_3d.dzt.isel(time=blocks[i]).load()
    
    bud = compute_mixed_layer_temperature_budget_offline(ds_day_budget_3d_reduced,ds_day_budget_3d_reduced_2d,mld,dzt)
    
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])

    temp_in_mld = (ds_day_snapshot.isel(time=blocks_snap[i]).temp_in_mld/rho0).load()
    bud = compute_tendency_entrainment(bud,temp_in_mld)

    #    mlt_budget_stavg_daily_offline_uncat.append(bud)
    bud.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_offline_2023_block%03d.nc' % i)

In [ ]:
# Concat and save to a single file:
ds_cat = []

for i in tqdm(range(73)):
    ds = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_offline_2023_block%03d.nc' % i).load()
    ds_cat.append(ds)
ds = xr.concat(ds_cat,dim='time')

ds.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_offline_2023.nc')

In [ ]:
# Create monthly means from daily files and save back to file:
fname = '/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_output365.nc'
ds  = xr.open_dataset(fname, chunks={'time':61, 'yt_ocean': 180, 'xt_ocean': 240})
ds_mon = xr.zeros_like(ds.resample(time='1ME').mean()).copy(deep=True)

for var in tqdm(ds.data_vars):
    ds_mon[var] = ds[var].resample(time='1ME').mean().load()
ds_mon = ds_mon.assign_coords({'time':ds.time.resample(time='1ME').mean()})
ds_mon.to_netcdf(fname[:-3] + '_monthly_mean.nc')

#### Spatial plots comparing monthly, daily offline and daily:

In [ ]:
# Load from file:
mlt_budget_stavg_monthly_offline = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_monthly_offline_output370.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()
mlt_budget_stavg_daily_offline  = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_offline_output370_monthly_mean.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()
mlt_budget_stavg_daily_online  = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_output370_monthly_mean.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()

In [ ]:
regs = {'Warm Pool':[-200, -160, -10, 10],
        'Tasman Sea':[-205, -190, -40, -25],
        'Cold Tongue':[-150,-90,-5,5],
       }

In [ ]:
# Region selection:
fig = plt.figure(figsize=(30,15))
mlt_budget_stavg_monthly_offline['vert_mixing'].mean('time').plot()
for key in regs.keys():
    sreg = regs[key]
    plt.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
plt.gca().set_facecolor('k')

In [ ]:
# Spatial plots:
fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(15,6),layout='constrained')

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
budgets = [mlt_budget_stavg_monthly_offline.mean('time'),
           mlt_budget_stavg_daily_offline.mean('time'),
           mlt_budget_stavg_daily_online.mean('time'),
           #mlt_budget_stavg_daily_online.mean('time') - mlt_budget_stavg_monthly_offline.mean('time'),
           #mlt_budget_stavg_daily_online.mean('time') - mlt_budget_stavg_daily_offline.mean('time')
          ]
budget_labels = ['Monthly Offline','Daily Offline','Online','Online - Monthly Offline','Online - Daily Offline']
unit_conv = 86400*30.5
clims = [1,1,1,2.5,5,5]

for i in range(len(budgets)):
    for j, var in enumerate(vars):
        if i == 0:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',cbar_kwargs={'label':labels[j] + ' ($^\circ$C/month)','shrink':0.7,'location':'top'})
        else:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',add_colorbar=False)
        axs[i][j].set_ylabel('')
    axs[i][0].set_ylabel(budget_labels[i])
    
for key in regs.keys():
    sreg = regs[key]
    axs[2][0].plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k',linewidth=0.5)

for ax in axs.reshape(-1):
    ax.set_ylim([-60,60])
    ax.set_xlabel('')
    ax.set_title('')
    ax.set_facecolor('k')
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.savefig('MLT_budget_2023_Online_Offline_Comparison_Pacific.png',dpi=300)

In [ ]:
# Compute spatial averages:

budgets = {}

def area_average(budget,reg):

    total_area = ds_grid.area_t.sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])
    budget_av = (budget*ds_grid.area_t).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])/total_area
    return(budget_av)

for key in tqdm(regs.keys()):
    sreg = regs[key]
    budgets[key] = [area_average(mlt_budget_stavg_monthly_offline,sreg).load(),
               area_average(mlt_budget_stavg_daily_offline,sreg).load(),
               area_average(mlt_budget_stavg_daily_online,sreg).load()
              ]

In [ ]:
# Time series across specific regions:
fig, axs = plt.subplots(nrows=len(regs.keys()), ncols=1, figsize=(12,12))

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
cols = ['k','r','b','g','m','c']
typs= ['-','--',':']
budget_labels = ['Monthly Offline','Daily Offline','Online']
unit_conv = 86400*30.5

for k, key in enumerate(regs.keys()):
    sreg = regs[key]
    for i in range(len(budgets)):
        for j, var in enumerate(vars):
            if j == 0 and k == 0:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=budget_labels[i])
            elif i == 0 and k == 1:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=labels[j])
            else:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2)
    axs[k].set_title(key)
axs[0].legend()
axs[1].legend()
plt.tight_layout()
plt.savefig('MLT_budget_2023_Online_Offline_Comparison_Time_Series.png',dpi=300)

In [ ]:

fig, axes = plt.subplots(nrows=3,ncols=1,figsize=(13, 15))

months = {'Warm Pool':range(12),
        'Tasman Sea':[9,10,11,0,1,2],
        'Cold Tongue':range(12),
       }
month_lab = {'Warm Pool':'Annual',
        'Tasman Sea':'Oct-Dec, Jan-Feb',
        'Cold Tongue':'Annual'}

# Bar plots across specific regions:
for i, key in enumerate(budgets.keys()):
    ax = axes[i]
    
    ds = xr.concat(budgets[key],dim='class').assign_coords({'class':['Monthly Offline','Daily Offline','Online']}).isel(time=months[key]).mean('time')
    
    # Add another variable:
    ds['surface_flux_total'] = ds['surface_flux']+ds['sw_pen']
    ds['vert_total'] = ds['surface_flux']+ds['sw_pen']+ds['vert_mixing']
    
    variables = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen','residual','surface_flux_total','vert_total']
    classes = ['Monthly Offline','Daily Offline','Online']
    labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration','Residual','Surface fluxes +\nSW penetration','Surface fluxes+ \nSW penetration \n+ Vertical Mixing']
    
    # Convert to DataFrame for easier plotting
    df = ds.to_dataframe()
    
    # Reset index to get 'class' as a column
    df = df.reset_index()
    
    # Parameters
    num_vars = len(variables)
    num_classes = len(classes)
    bar_width = 0.25
    x = np.arange(num_vars)  # One x position per variable
    
    unit_conv = 86400*30.5
    # Create figure
    
    # Plot each class as a separate bar group
    for i, cls in enumerate(classes):
        # Get values for this class across all variables
        values = [df[df['class'] == cls][var].values[0]*unit_conv for var in variables]
        
        # Offset x positions for each class
        ax.bar(x + i * bar_width, values, width=bar_width, label=cls)
    
    # Formatting
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(labels)
    ax.set_ylabel("$^\circ$C/month")
    ax.set_title(key + ' Mixed Layer Temperature Budget ' +month_lab[key] + ' 2023')
    ax.legend(title="Budget")
    ax.grid()
    plt.tight_layout()
plt.savefig('MLT_budget_Online_Offline_Comparison_Bar.png',dpi=250)

# Testing and checks

## Check monthly accumulations

There is a bug in the accumulation of the monthly _in_mld budget terms. It only seems to affect the files in "ocean_budget_month.nc". The other files (e.g. "ocean_budget_month_3d.nc", when compared to "ocean_budget_daily_3d.nc", and "ocean_month.nc", when compared to "ocean_daily.nc" seem fine).

I don't know where it is coming from. It's weird... not really sure how to fix it either, but for now we just don't output any ocean_budget_month.nc terms...


In [ ]:
ds_day = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = xr.open_dataset(base2 + 'ocean_budget_month.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
var = 'temp_vdiffuse_diff_cbt_in_mld'
ds_day = xr.open_dataset(base2 + 'ocean_budget_daily_3d.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).isel(st_ocean=0)
ds_mon = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).isel(st_ocean=0)
var = 'temp_vdiffuse_diff_cbt'
# ds_day = xr.open_dataset(base2 + 'ocean_daily.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# ds_mon = xr.open_dataset(base2 + 'ocean_month.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# var = 'pme_river';

fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(18,5))
ds_day[var].mean('time').plot(ax=axes[0])
ds_mon[var].isel(time=0).plot(ax=axes[1])
(ds_day[var].mean('time') - ds_mon[var].isel(time=0)).plot(ax=axes[2])
plt.tight_layout()

## Test that the free-surface equation closes:

In [ ]:
ds_month = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).isel(time=0)

In [ ]:
# Compute convU from transports:
Fx = ds_month.tx_trans_int_z/rho0
Fy = ds_month.ty_trans_int_z/rho0
Ah = ds_grid.area_t
dFxdx = xr.concat([Fx.isel(xu_ocean=-1),Fx],dim='xu_ocean').diff('xu_ocean').rename({'xu_ocean':'xt_ocean'}).assign_coords(xt_ocean=ds_month.xt_ocean.values)/Ah # longitude is cylic
dFydy = xr.concat([Fy.isel(yu_ocean=0),Fy],dim='yu_ocean').diff('yu_ocean').rename({'yu_ocean':'yt_ocean'}).assign_coords(yt_ocean=ds_month.yt_ocean.values)/Ah # Latitude - note that the first element will be wrong, but it's inside Antarctica so doesn't matter
convU = -(dFxdx + dFydy)

# convU directly:
convU_direct = ds_month.conv_rho_ud_t/rho0

# Other terms (all ms-1):
detadt = ds_month.eta_t_tendency
pme = ds_month.pme_river/rho0
eta_smoother = ds_month.eta_smoother
res = detadt - pme - convU_direct - eta_smoother

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))

detadt.plot(ax=axes[0][0],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][0].set_title('deta/dt')
pme.plot(ax=axes[0][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][1].set_title('P-E+R')
eta_smoother.plot(ax=axes[0][2],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][2].set_title('eta-smoother')
convU.plot(ax=axes[1][0],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[1][0].set_title('convU (from transport convergence)')
convU_direct.plot(ax=axes[1][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[1][1].set_title('convU (from conv_rho_ud_t)')
res.plot(ax=axes[1][2],vmin=-1.e-10,vmax=1.e-10,cmap='RdBu_r')
axes[1][2].set_title('deta/dt - PME - convU (from conv_rho_ud_t)- eta-smoother')
plt.savefig('Free_Surface_Budget_Closure.png',dpi=150)

## Test free-surface equation times tracer_in_mld:

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_month = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

#detadt = ds.eta_t_tendency_times_temp_in_mld.mean('time')
#pme = ds.pme_river_times_temp_in_mld.mean('time')
#eta_smoother = ds.eta_smoother_times_temp_in_mld.mean('time')

detadt = ds.eta_t_tendency_times_salt_in_mld.mean('time')
pme = ds.pme_river_times_salt_in_mld.mean('time')
eta_smoother = ds.eta_smoother_times_salt_in_mld.mean('time')

convU = detadt - pme - eta_smoother

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(15,10))
vmin = -0.0002
vmax = 0.0002
detadt.plot(ax=axes[0][0],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[0][0].set_title('deta/dt * ML temperature')
pme.plot(ax=axes[0][1],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[0][1].set_title('P-E+R * ML temperature')
eta_smoother.plot(ax=axes[1][0],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[1][0].set_title('eta-smoother * ML temperature')
convU.plot(ax=axes[1][1],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[1][1].set_title('convU * ML temperature')
#plt.savefig('Free_Surface_Budget_Times_MLT.png',dpi=150)

## Test tracer_at_mlb diagnostics

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_month = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

mlt = ds.temp_in_mld/rho0
tmlb = ds.temp_at_mlb
mls = ds.salt_in_mld/rho0
smlb = ds.salt_at_mlb

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
mlt.isel(time=0).plot(ax=axes[0][0],vmin=10.,vmax=30.,cmap='RdBu_r')
axes[0][0].set_title('ML temperature')
tmlb.isel(time=0).plot(ax=axes[1][0],vmin=10.,vmax=30.,cmap='RdBu_r')
axes[1][0].set_title('Temperature at MLB')
(mlt-tmlb).isel(time=0).plot(ax=axes[2][0],vmin=-.5,vmax=.5,cmap='RdBu_r')
axes[2][0].set_title('Difference')
mls.isel(time=0).plot(ax=axes[0][1],vmin=33.,vmax=36.,cmap='RdBu_r')
axes[0][1].set_title('ML salinity')
smlb.isel(time=0).plot(ax=axes[1][1],vmin=33.,vmax=36.,cmap='RdBu_r')
axes[1][1].set_title('Salinity at MLB')
(mls-smlb).isel(time=0).plot(ax=axes[2][1],vmin=-.1,vmax=.1,cmap='RdBu_r')
axes[2][1].set_title('Difference')
plt.savefig('ML_and_MLB_tracers.png',dpi=150)

## Test tracer_at_mlb correction diagnostics

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds['eta_t_tendency_times_temp_in_mld']/rho0).isel(time=0).plot(ax=axes[0][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[0][0].set_title('eta_t_tendency_times_temp_in_mld')
(ds['eta_t_tendency_times_temp_at_mlb']/rho0).isel(time=0).plot(ax=axes[1][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[1][0].set_title('eta_t_tendency_times_temp_at_mlb')
(ds['s_surf_ent_temp']/rho0).isel(time=0).plot(ax=axes[2][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[2][0].set_title('s_surf_ent_temp')
(ds['eta_t_tendency_times_temp_in_mld']/rho0-ds['eta_t_tendency_times_temp_at_mlb']/rho0).isel(time=0).plot(ax=axes[0][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][1].set_title('eta_t_tendency_times_temp_in_mld - eta_t_tendency_times_temp_at_mlb')
(ds['eta_t_tendency_times_temp_at_mlb']/rho0-ds['s_surf_ent_temp']/rho0).isel(time=0).plot(ax=axes[1][1],vmin=-.5e-6,vmax=.5e-6,cmap='RdBu_r')
axes[1][1].set_title('eta_t_tendency_times_temp_at_mlb - s_surf_ent_temp')

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds['eta_t_tendency_times_salt_in_mld']/rho0).isel(time=0).plot(ax=axes[0][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[0][0].set_title('eta_t_tendency_times_salt_in_mld')
(ds['eta_t_tendency_times_salt_at_mlb']/rho0).isel(time=0).plot(ax=axes[1][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[1][0].set_title('eta_t_tendency_times_salt_at_mlb')
(ds['s_surf_ent_salt']/rho0).isel(time=0).plot(ax=axes[2][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[2][0].set_title('s_surf_ent_salt')
(ds['eta_t_tendency_times_salt_in_mld']/rho0-ds['eta_t_tendency_times_salt_at_mlb']/rho0).isel(time=0).plot(ax=axes[0][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][1].set_title('eta_t_tendency_times_salt_in_mld - eta_t_tendency_times_salt_at_mlb')
(ds['eta_t_tendency_times_salt_at_mlb']/rho0-ds['s_surf_ent_salt']/rho0).isel(time=0).plot(ax=axes[1][1],vmin=-.5e-6,vmax=.5e-6,cmap='RdBu_r')
axes[1][1].set_title('eta_t_tendency_times_salt_at_mlb - s_surf_ent_salt')

#plt.savefig('ML_and_MLB_tracers.png',dpi=150)

## Plot correction terms and what they correct

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

In [ ]:
fig, axes = plt.subplots(nrows=4,ncols=3,figsize=(20,20))
axs = axes.reshape(-1)
vars = {'$C_a Q_m / \\rho_0 H$':ds_day_budget['sfc_hflux_pme_in_mld']/rho0/Cp,
        '$-C_H Q_m / \\rho_0 H$':-ds_day_budget['pme_river_times_temp_in_mld']/rho0,
        '$(C_a - C_H) Q_m / \\rho_0 H$':ds_day_budget['sfc_hflux_pme_in_mld']/rho0/Cp-ds_day_budget['pme_river_times_temp_in_mld']/rho0,
        '$C_{adv}/\\rho_0/H$':ds_day_budget['temp_advection_in_mld']/rho0/Cp,
        '$C_H\\nabla\\cdot U / H$':(-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0,
        '$C_{ent} w_H^{(s)}/H$':ds_day_budget['s_surf_ent_temp']/rho0,
        '$C_H\\nabla\\cdot U / H + C_{ent} w_H^{(s)}/H$':(-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0+ds_day_budget['s_surf_ent_temp']/rho0,
        '$C_{adv}/\\rho_0/H + C_H\\nabla\\cdot U / H + C_{ent} w_H^{(s)}/H$':ds_day_budget['temp_advection_in_mld']/rho0/Cp + (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0 + ds_day_budget['s_surf_ent_temp']/rho0,
        'eta_smoother':ds_day_budget['temp_eta_smooth_in_mld']/rho0/Cp,
        'eta_smoother_cor':-ds_day_budget['eta_smoother_times_temp_in_mld']/rho0,
        'eta_smoother + eta_smoother_cor':ds_day_budget['temp_eta_smooth_in_mld']/rho0/Cp-ds_day_budget['eta_smoother_times_temp_in_mld']/rho0,
       }
for i, key in enumerate(vars.keys()):
    (vars[key].isel(time=0)*86400).plot(ax=axs[i],vmin=-.1,vmax=.1,cmap='RdBu_r',cbar_kwargs={'label':'degC/day'})
    #(vars[key].mean('time')*86400*31).plot(ax=axs[i],vmin=-.5,vmax=.5,cmap='RdBu_r',cbar_kwargs={'label':'degC/month'})
    axs[i].set_title(key)
#plt.savefig('Correction_terms_all_1989-01.png',dpi=150,bbox_inches='tight')

## Compare MLD grouped budget for one month with and without advection/PME correction terms:

In [ ]:
# With corrections:
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_with_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_with_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_with_cor,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily_with_cor.load();

In [ ]:
# Without corrections:
bud_tendency = 'temp_tendency_in_mld'
bud_var_grps = {'advection':['temp_advection_in_mld',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld',
                                'temp_eta_smooth_in_mld'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}
#ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_without_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_without_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_without_cor,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily_without_cor.load();

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=6,figsize=(35,15))
vars = ['mlt_tendency','entrainment','fixedh_tendency','advection','surface_flux','residual']
clim = .05
for i, var in enumerate(vars):
    (mlt_budget_stavg_daily_with_cor[var]*86400).mean('time').plot(ax=axes[0][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[0][i].set_title(var + ' (corrected)')
    (mlt_budget_stavg_daily_without_cor[var]*86400).mean('time').plot(ax=axes[1][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[1][i].set_title(var + ' (uncorrected)')
    ((mlt_budget_stavg_daily_with_cor[var]-mlt_budget_stavg_daily_without_cor[var])*86400).mean('time').plot(ax=axes[2][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[2][i].set_title(var + ' (difference)')
plt.savefig('Budget_Corrections_Check_1989-01.png',dpi=100,bbox_inches='tight')

## Compare MLD grouped budget for one month with and without advection/PME correction terms - Salinity

In [ ]:
# With corrections:
bud_tendency = 'salt_tendency_in_mld_cor'
bud_var_grps = {'advection':['salt_advection_in_mld_cor',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',
                                'pme_in_mld_cor',
                                'salt_eta_smooth_in_mld_cor']}
bud_var_extras = {}
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_with_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_with_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_with_cor,ds_day_snapshot.salt_in_mld/rho0)
mlt_budget_stavg_daily_with_cor.load();

In [ ]:
# Without corrections:
bud_tendency = 'salt_tendency_in_mld'
bud_var_grps = {'advection':['salt_advection_in_mld',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',
                                'salt_eta_smooth_in_mld']}
bud_var_extras = {}
#ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_without_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_without_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_without_cor,ds_day_snapshot.salt_in_mld/rho0)
mlt_budget_stavg_daily_without_cor.load();

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=6,figsize=(35,15))
vars = ['mlt_tendency','entrainment','fixedh_tendency','advection','surface_flux','residual']
clim = .02
for i, var in enumerate(vars):
    (mlt_budget_stavg_daily_with_cor[var]*86400).mean('time').plot(ax=axes[0][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[0][i].set_title(var + ' (corrected)')
    (mlt_budget_stavg_daily_without_cor[var]*86400).mean('time').plot(ax=axes[1][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[1][i].set_title(var + ' (uncorrected)')
    ((mlt_budget_stavg_daily_with_cor[var]-mlt_budget_stavg_daily_without_cor[var])*86400).mean('time').plot(ax=axes[2][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[2][i].set_title(var + ' (difference)')
plt.savefig('Budget_Corrections_Check_1989-01_Salinity.png',dpi=100)

## Test that Eulerian budget (no MLD binning) closes:

In [ ]:
# Choose time, region, depth:
time = 0; reg_slice= [-300, 300,-90,90]; st_ocean = 40;

# Load budget:
ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time).isel(st_ocean=st_ocean)

# List the terms:
bud_vars = ['temp_advection','temp_submeso','neutral_diffusion_temp','neutral_gm_temp','temp_vdiffuse_k33',
            'temp_nonlocal_KPP','temp_vdiffuse_diff_cbt',
            'temp_rivermix','temp_vdiffuse_sbc', 'frazil_3d', 
            'sw_heat']

# Add the 2D terms if we're looking at the surface layer:
if st_ocean == 0:
    bud_vars = bud_vars + ['temp_eta_smooth','sfc_hflux_pme']

# Compute the residual:
ds_day_budget_slice['residual'] = ds_day_budget_slice.temp_tendency.load().copy(deep=True)
for var in bud_vars:
    ds_day_budget_slice['residual'] -= ds_day_budget_slice[var].load()

# Add tendency and residual to the terms list:
bud_vars = ['temp_tendency','residual'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=4,figsize=(25,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-2):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-10.,vmax=10.,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that MLD binned budgets (standard and falling averages) close:

In [ ]:
# Choose time and region:
time = 0; reg_slice = [-300, 300,-90,90]#reg = [-100, 20, 0, 60]

# List the terms:
bud_vars = ['temp_tendency_in_mld',
            'sfc_hflux_pme_in_mld','temp_eta_smooth_in_mld','temp_advection_in_mld','temp_submeso_in_mld','neutral_diffusion_in_mld_temp','neutral_gm_in_mld_temp','temp_vdiffuse_k33_in_mld',
            'temp_nonlocal_KPP_in_mld','temp_vdiffuse_diff_cbt_in_mld',
            'temp_rivermix_in_mld','temp_vdiffuse_sbc_in_mld', 'frazil_3d_in_mld', 
            'sw_heat_in_mld']

# Load budget:
# Standard average:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)
ds_day_budget_slice = ds_day_budget[bud_vars]

# Falling average:
# ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc')[bud_vars].sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)/86400.

# Compute residual:
ds_day_budget_slice['residual_in_mld'] = ds_day_budget_slice.temp_tendency_in_mld.load().copy(deep=True)
for var in bud_vars[1:]:
    ds_day_budget_slice['residual_in_mld'] -= ds_day_budget_slice[var].load()

# Add residual to budget list:
bud_vars = ['residual_in_mld'] + bud_vars

# Add correction terms for plotting:
ds_day_budget_slice['adv_cor1'] = ((-ds_day_budget['eta_t_tendency_times_temp_in_mld']/rho0+ds_day_budget['pme_river_times_temp_in_mld']/rho0 + ds_day_budget['eta_smoother_times_temp_in_mld']/rho0)*rho0*Cp).load()
ds_day_budget_slice['adv_cor2'] = (ds_day_budget['s_surf_ent_temp']*Cp).load()
ds_day_budget_slice['pme_cor'] = ((-ds_day_budget['pme_river_times_temp_in_mld']/rho0)*rho0*Cp).load()
ds_day_budget_slice['eta_smoother_cor'] = (-ds_day_budget['eta_smoother_times_temp_in_mld']*Cp).load()

# Add terms to budget list:
bud_vars = ['adv_cor1','adv_cor2','pme_cor','eta_smoother_cor'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=5,figsize=(30,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-3):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-1,vmax=1,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that Eulerian budget (no MLD binning) closes - Salinity (NO DATA!!!):

In [ ]:
# Choose time, region, depth:
time = 0; reg_slice= [-300, 300,-90,90]; st_ocean = 40;

# Load budget:
ds_mon_budget_slice = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time).isel(st_ocean=st_ocean)

# List the terms:
bud_vars = ['salt_advection','salt_submeso','neutral_diffusion_salt','neutral_gm_salt','salt_vdiffuse_k33',
            'salt_nonlocal_KPP','salt_vdiffuse_diff_cbt',
            'salt_rivermix','salt_vdiffuse_sbc']

# Add the 2D terms if we're looking at the surface layer:
if st_ocean == 0:
    bud_vars = bud_vars + ['salt_eta_smooth']

# Compute the residual:
ds_day_budget_slice['residual'] = ds_day_budget_slice.salt_tendency.load().copy(deep=True)
for var in bud_vars:
    ds_day_budget_slice['residual'] -= ds_day_budget_slice[var].load()

# Add tendency and residual to the terms list:
bud_vars = ['salt_tendency','residual'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=4,figsize=(25,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-2):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-10.,vmax=10.,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that MLD binned budgets (standard and falling averages) close - Salinity:

In [ ]:
# Choose time and region:
time = 0; reg_slice = [-300, 300,-90,90]#reg = [-100, 20, 0, 60]

# List the terms:
bud_vars = ['salt_tendency_in_mld','salt_eta_smooth_in_mld','salt_advection_in_mld','salt_submeso_in_mld','neutral_diffusion_in_mld_salt','neutral_gm_in_mld_salt','salt_vdiffuse_k33_in_mld',
            'salt_nonlocal_KPP_in_mld','salt_vdiffuse_diff_cbt_in_mld',
            'salt_rivermix_in_mld','salt_vdiffuse_sbc_in_mld']

# Load budget:
# Standard average:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)
ds_day_budget_slice = ds_day_budget[bud_vars]

# Falling average:
# ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc')[bud_vars].sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)/86400.

# Compute residual:
ds_day_budget_slice['residual_in_mld'] = ds_day_budget_slice.salt_tendency_in_mld.load().copy(deep=True)
for var in bud_vars[1:]:
    ds_day_budget_slice['residual_in_mld'] -= ds_day_budget_slice[var].load()

# Add residual to budget list:
bud_vars = ['residual_in_mld'] + bud_vars

# Add correction terms for plotting:
#ds_day_budget_slice['convU_correction'] = ((ds_day_budget['eta_t_tendency_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.).load()
ds_day_budget_slice['convU_correction'] = ((ds_day_budget['eta_t_tendency_times_salt_in_mld']-ds_day_budget['pme_river_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.).load()
ds_day_budget_slice['pme_correction'] = (ds_day_budget['pme_river_times_salt_in_mld']/1000.).load()

# Add terms to budget list:
bud_vars = ['convU_correction','pme_correction'] + bud_vars

## Units:

# salt_in_mld = psu kg m-3 = g m-3
# eta_t_tendency = m s-1
# eta_t_tendency_times_salt_in_mld = g m-3 m s-1 m-1 = g m-3 s-1 = psu kg m-3 s-1

# salt_tendency_in_mld = kg m-3 s-1

# to convert eta_t_tendency_times_salt_in_mld to salt_tendency_in_mld units, divide by 1000 (g/kg)

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=5,ncols=4,figsize=(25,25))
axs = axes.reshape(-1)
clim = .25e-6
print('Spatial maximums of terms (1e-6 kg m-3 s-1):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values/1.e-6) + ' ' + var)


In [ ]:
# Plot correction terms and what they correct:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))

clim = .25e-6

ds_day_budget_slice['salt_advection_in_mld'].plot(ax=axes[0][0],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[0][0].set_title('salt_advection_in_mld')
ds_day_budget_slice['convU_correction'].plot(ax=axes[0][1],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[0][1].set_title('convU_correction')
(ds_day_budget_slice['salt_advection_in_mld']-ds_day_budget_slice['convU_correction']).plot(ax=axes[0][2],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[0][2].set_title('salt_advection_in_mld - convU_correction')
#ds_day_budget_slice['sfc_hflux_pme_in_mld'].plot(ax=axes[1][0],vmin=-clim,vmax=clim,cmap='RdBu_r')
#axes[1][0].set_title('sfc_hflux_pme_in_mld')
ds_day_budget_slice['pme_correction'].plot(ax=axes[1][1],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[1][1].set_title('pme_correction')
(-ds_day_budget_slice['pme_correction']).plot(ax=axes[1][2],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[1][2].set_title('- pme_correction')
plt.savefig('Correction_Terms_salinity.png',dpi=150)

## Bug in temp_mld diagnostic demonstration (see https://github.com/mom-ocean/MOM5/issues/397):

In [ ]:
fig,axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds_month.mld).isel(time=0).plot(vmin=0.,vmax=100.,ax=axes[0][0],cmap=cm.cm.amp)
axes[0][0].set_title('MLD [m]')
(ds_month.temp_mld/ds_day.mld/rho0).isel(time=0).plot(vmin=0.,vmax=30,ax=axes[1][0],cmap=cm.cm.thermal)
axes[1][0].set_title('MLT from temp_mld [degC]')
(ds_month.temp_avg_mld/rho0).isel(time=0).plot(vmin=0.,vmax=30.,ax=axes[0][1],cmap=cm.cm.thermal)
axes[0][1].set_title('MLT from temp_avg_mld [degC]')
(ds_month.temp_in_mld/rho0).isel(time=0).plot(vmin=0.,vmax=30.,ax=axes[1][1],cmap=cm.cm.thermal)
axes[1][1].set_title('MLT from temp_in_mld [degC]')
(ds_month.temp-273.15).isel(time=0,st_ocean=0).plot(vmin=0.,vmax=30.,ax=axes[2][0],cmap=cm.cm.thermal)
axes[2][0].set_title('SST [degC]')
(ds_month.temp.isel(st_ocean=0)-273.15 - ds_month.temp_in_mld/rho0).isel(time=0).plot(vmin=-0.05,vmax=0.05,ax=axes[2][1],cmap='RdBu_r')
axes[2][1].set_title('SST - MLT from temp_in_mld [degC]')
plt.savefig('temp_mld_diagnostic_checks_global.png',dpi=250)

## Hat-averaging, single-output tendencies check

In [ ]:
# 3D diagnostics:
HC_tend_stavg = ds_day_budget.temp_tendency.isel(st_ocean=0)
HC_tend_falavg = ds_day_budget_falavg.temp_tendency.isel(st_ocean=0)/(ds_day_budget_falavg.average_DT/np.timedelta64(1,'s')) # Division by Dt in seconds required because falling average is an integral not a sum.
HC_tend_risavg = HC_tend_stavg - HC_tend_falavg # Rising average = standard average - falling average

# Time-averaged HC and tendency:
HC = ds_day_budget.temp_rhodzt.isel(st_ocean=0)*Cp
HC_tend_from_HC = HC.diff('time')/86400.
time_cen = [HC.time.isel(time=slice(x,x+2)).mean('time').values for x in range(len(HC.time))][:-1]
HC_tend_from_HC = HC_tend_from_HC.assign_coords({'time':time_cen})

# Time-snapshot HC and tendency:
HC_snap = (ds_day_snapshot.temp.isel(st_ocean=0)-273.15)*rho0*ds_day_snapshot.dzt.isel(st_ocean=0)*Cp
HC_snap_tend_from_HC = HC_snap.diff('time')/86400.
HC_snap_tend_from_HC = HC_snap_tend_from_HC.assign_coords({'time':HC.time.values})

# Time-averaged tendency from hat average:
HC_tend_hatavg = xr.zeros_like(HC_tend_hatavg_from_HC)
HC_tend_hatavg.data = HC_tend_risavg.isel(time=slice(0,-1)).values + HC_tend_falavg.isel(time=slice(1,None)).values

In [ ]:
# Plot at a point
xt = 20
yt = 1
times = slice(0,14)

fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(14,10))

HC_snap.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][0],label='Snapshot HC')
HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][0],label='Daily-averaged HC')
HC_snap.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][1],label='Snapshot HC')
HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][1],label='Daily-averaged HC')

HC_snap_tend_from_HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='d HC_snap/dt',linewidth=3.)
HC_tend_from_HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][1],label='d HC/ dt',linewidth=3.)
HC_tend_hatavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][1],label='Tendency, hat average')
HC_tend_stavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, standard average')
HC_tend_risavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, rising average')
HC_tend_falavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, falling average')
for ax in axes.reshape(-1):
    ax.grid()
    ax.set_xlim(axes[0][0].get_xlim())
    ax.legend()

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))
index = 1

HC_tend_stavg.isel(time=index).plot(ax=axes[0][0],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][0].set_title('Tendency standard average (Wm-2)')
HC_tend_risavg.isel(time=index).plot(ax=axes[0][1],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][1].set_title('Tendency rising average (Wm-2)')
HC_tend_falavg.isel(time=index).plot(ax=axes[0][2],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][2].set_title('Tendency falling average (Wm-2)')
HC_tend_from_HC.isel(time=0).plot(ax=axes[1][0],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[1][0].set_title('Tendency from daily-average HC (Wm-2)')
HC_tend_hatavg.isel(time=0).plot(ax=axes[1][1],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[1][1].set_title('Tendency hat average (Wm-2)')
(HC_tend_from_HC - HC_tend_hatavg).isel(time=0).plot(ax=axes[1][2])
axes[1][2].set_title('Difference')

## Hat-averaging, combined-outputs epoch difference check

In [ ]:
# single-output standard, rising and falling averages:
HC_tend_stavg = ds_day_budget.temp_tendency.isel(st_ocean=0)*(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Short period standard difference (hence x DT)
HC_tend_falavg = ds_day_budget_falavg.temp_tendency.isel(st_ocean=0)                                        # Short period falling average difference (already x DT in code)
HC_tend_risavg = HC_tend_stavg - HC_tend_falavg                                                             # Short period rising average difference (standard - falling)

# Time-averaged heat content and its single-output tendency:
HC = ds_day_budget.temp_rhodzt.isel(st_ocean=0)*Cp
HC_tend_from_HC = HC.diff('time')/86400.
time_cen = [HC.time.isel(time=slice(x,x+2)).mean('time').values for x in range(len(HC.time))][:-1]
HC_tend_from_HC = HC_tend_from_HC.assign_coords({'time':time_cen})

In [ ]:
# Define epoch periods:
epoch1 = list(range(10))
epoch2 = [len(HC.time) - 5 + x for x in range(5)]
epochM = np.arange(epoch1[-1]+1,epoch2[0],1)

n_minus_1_epoch1 = xr.DataArray(data=range(len(epoch1)),dims=['time'],coords={'time':HC_tend_stavg.time.isel(time=epoch1)}) # (n-1) for epoch1 as a DataArray
n_minus_1_epoch2 = xr.DataArray(data=range(len(epoch2)),dims=['time'],coords={'time':HC_tend_stavg.time.isel(time=epoch2)}) # (n-1) for epoch2 as a DataArray

# Compute epoch HC change from HC:
HC_change = HC.isel(time=epoch2).mean('time') - HC.isel(time=epoch1).mean('time') # All days have same length, so no weighted mean is needed

# Compute epoch HC change from hat average tendencies:
long_ris_epoch1 = HC_tend_risavg.isel(time=epoch1).mean('time') + (HC_tend_stavg.isel(time=epoch1)*n_minus_1_epoch1).mean('time')
long_sta_epoch1 = HC_tend_stavg.isel(time=epoch1).sum('time')
long_fal_epoch1 = long_sta_epoch1 - long_ris_epoch1

long_sta_epochM = HC_tend_stavg.isel(time=epochM).sum('time')

long_ris_epoch2 = HC_tend_risavg.isel(time=epoch2).mean('time') + (HC_tend_stavg.isel(time=epoch2)*n_minus_1_epoch2).mean('time')
long_sta_epoch2 = HC_tend_stavg.isel(time=epoch2).sum('time')
long_fal_epoch2 = long_sta_epoch2 - long_ris_epoch2

HC_change_from_hatavg = long_ris_epoch1 + long_sta_epochM + long_fal_epoch2

In [ ]:
# Plot at a point
xt = 20
yt = 1
times = slice(0,len(HC.time))

fig = plt.figure(figsize=(9,5))
axes = [plt.gca()]

HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0],label='Daily-averaged HC')
axes[0].plot(HC.time.isel(time=epoch1),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1).mean('time').values*xr.ones_like(HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1)),linewidth=3.,color='C0')
axes[0].text(HC.time.isel(time=epoch1[0]),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1).mean('time').values,'Epoch 1')
axes[0].plot(HC.time.isel(time=epoch2),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2).mean('time').values*xr.ones_like(HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2)),linewidth=3.,color='C0')
axes[0].text(HC.time.isel(time=epoch2[0]),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2).mean('time').values,'Epoch 2')

axes[0].text(np.datetime64('2023-06-01'),0.6e7,'%5.0f = Epoch 1 rising' % long_ris_epoch1.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.65e7,'%5.0f = Epoch M standard' % long_sta_epochM.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.7e7,'%5.0f = Epoch 2 falling' % long_fal_epoch2.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.75e7,'%5.0f = HC change from hat' % HC_change_from_hatavg.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.8e7,'%5.0f = HC change' % HC_change.isel(xt_ocean=xt,yt_ocean=yt).values)

for ax in axes:
    ax.grid()
    ax.set_xlim(axes[0].get_xlim())
    ax.legend()

In [ ]:
# Plot spatial slice:
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(20,6))

HC_change.plot(ax=axes[0],vmin=-1.e7,vmax=1.e7,cmap='RdBu_r')
axes[0].set_title('HC change (Jm-2)')
HC_change_from_hatavg.plot(ax=axes[1],vmin=-1.e7,vmax=1.e7,cmap='RdBu_r')
axes[1].set_title('HC change from hat avg (Jm-2)')
(HC_change_from_hatavg-HC_change).plot(ax=axes[2],vmin=-1.e2,vmax=1.e2,cmap='RdBu_r')
axes[2].set_title('Difference')

##### Some old code from Chris that I probably don't need

In [ ]:
# Use a hack of Chris's code to do the hatavg from tendency calculation:

# First, get times (in seconds):
time_stamp_file = base2 + 'time_stamp.out'

def init_run_time(time_stamp_file):
    with open(time_stamp_file, "r") as tstamp:
        tls = tstamp.readline().split()
        tls = [int(_) for _ in tls[:-1]]
        t0_date = datetime.datetime(tls[0], tls[1], tls[2], hour=tls[3], minute=tls[4], second=tls[5])
    time_units = datetime.datetime(1,1,1,0,0,0)
    return float((t0_date - time_units).total_seconds())

init_time = init_run_time(time_stamp_file)
init_times = init_time + (ds_day_budget.average_DT/np.timedelta64(1,'s')).cumsum() - 86400.
final_times = init_time + (ds_day_budget.average_DT/np.timedelta64(1,'s')).cumsum()

# Second,
# def compute_long_ris_avg_fwd(oheat_diag, ora_diag, monthly_avs, final_month_times):
#     """oheat_diag: time-average diagnostic
#     ora_diag: rising average diagnostic
#     monthly_avs: length of month
#     final month times: array of the last timestep of each averaging period (including timestep before run)"""
#     temp_tend_stnd_av = (oheat_diag*monthly_avs).sum('time')/(365*60*60*24) # 1/N * sum(std_av*month)
#     t_weight_av = ora_diag - oheat_diag*final_month_times 
#     t_weight_av_total = (t_weight_av*monthly_avs/(365*60*60*24)).sum('time') + final_month_times[-1]*temp_tend_stnd_av 
#     return t_weight_av_total

risavg_fwd_t0 = HC_tend_risavg.isel(time=0) - HC_tend_stavg.isel(time=0)*final_time.isel(time=0).values + final_time.isel(time=0).values*HC_tend_stavg.isel(time=0)
# def compute_long_ris_avg_bwd(oheat_diag, ora_diag, monthly_avs, initial_month_times):
#     """oheat_diag: time-average diagnostic
#     ora_diag: rising average diagnostic
#     monthly_avs: length of month
#     initial month times: array of the first timestep of each averaging period (including 1st timestep)"""
#     temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
#     t_weight_av = ora_diag + oheat_diag*initial_month_times # ris_avg + std_av*init_times
#     # m/N*(ris_avg + std_av*init_times) - t1*total_av
#     t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) - initial_month_times[0]*temp_tend_stnd_av 
#     return t_weight_av_total

risavg_bwd_t1 = (HC_tend_risavg.isel(time=slice(0,2)) + HC_tend_stavg.isel(time=slice(0,2))*init_times.isel(time=slice(0,2))).mean('time') - init_times[-1]*HC_tend_stavg.isel(time=slice(0,2)).mean('time')


In [ ]:
# Chris's code:
# compute long period rising average
def compute_long_ris_avg_fwd(oheat_diag, ora_diag, monthly_avs, final_month_times):
    """oheat_diag: time-average diagnostic
    ora_diag: rising average diagnostic
    monthly_avs: length of month
    final month times: array of the last timestep of each averaging period (including timestep before run)"""
    temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
    t_weight_av = ora_diag - oheat_diag*final_month_times 
    t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) + final_month_times[-1]*temp_tend_stnd_av 
    return t_weight_av_total

def compute_long_ris_avg_bwd(oheat_diag, ora_diag, monthly_avs, initial_month_times):
    """oheat_diag: time-average diagnostic
    ora_diag: rising average diagnostic
    monthly_avs: length of month
    initial month times: array of the first timestep of each averaging period (including 1st timestep)"""
    temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
    t_weight_av = ora_diag + oheat_diag*initial_month_times # ris_avg + std_av*init_times
    # m/N*(ris_avg + std_av*init_times) - t1*total_av
    t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) - initial_month_times[0]*temp_tend_stnd_av 
    return t_weight_av_total

def init_run_time(time_stamp_file):
    with open(time_stamp_file, "r") as tstamp:
        tls = tstamp.readline().split()
        tls = [int(_) for _ in tls[:-1]]
        t0_date = datetime.datetime(tls[0], tls[1], tls[2], hour=tls[3], minute=tls[4], second=tls[5])
    time_units = datetime.datetime(1,1,1,0,0,0)
    return float((t0_date - time_units).total_seconds())